# GuidaPlate — USDA Foundation Foods Expansion
## Notebook 07
This notebook expands the curated food database from 50 Rwanda-specific 
foods to ~450 foods by adding USDA Foundation Foods (whole/raw foods only, 
no branded or processed items).

The original 50 Rwanda foods remain unchanged as the primary, trilingual, 
clinically-validated recommendation source. USDA additions are English-only 
and have ckd_stage_safe computed automatically from KDOQI 2020 potassium 
and phosphorus thresholds.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()

# Load existing 50-food database
existing = pd.read_csv(ROOT / 'backend' / 'data' / 'food_database.csv')
print(f"Existing database: {len(existing)} foods")
print(f"Columns: {list(existing.columns)}")

# Load USDA raw files
food = pd.read_csv(ROOT / 'data' / 'raw' / 'usda' / 'food.csv')
food_nutrient = pd.read_csv(ROOT / 'data' / 'raw' / 'usda' / 'food_nutrient.csv')
nutrient = pd.read_csv(ROOT / 'data' / 'raw' / 'usda' / 'nutrient.csv')

# Filter to Foundation Foods only
foundation = food[food['data_type'] == 'foundation_food'].copy()
print(f"\nFoundation Foods available: {len(foundation)}")

Existing database: 50 foods
Columns: ['food_id', 'english', 'french', 'kinyarwanda', 'category', 'meal_type', 'protein_g', 'potassium_mg', 'phosphorus_mg', 'sodium_mg', 'energy_kcal', 'preparation_method', 'source', 'ckd_stage_safe', 'notes']

Foundation Foods available: 436


/var/folders/dc/6rtlztbj1tq0t4yb63tgwm8w0000gn/T/ipykernel_40644/3245038683.py:14: DtypeWarning: Columns (0: footnote) have mixed types. Specify dtype option on import or set low_memory=False.
  food_nutrient = pd.read_csv(ROOT / 'data' / 'raw' / 'usda' / 'food_nutrient.csv')


In [2]:
# Nutrient IDs we need
NUTRIENT_IDS = {
    'potassium_mg': 1092,   # Potassium, K - MG
    'phosphorus_mg': 1091,  # Phosphorus, P - MG
    'protein_g': 1003,      # Protein - G
    'sodium_mg': 1093,      # Sodium, Na - MG
    'energy_kcal': 1008,    # Energy - KCAL
}

# Filter food_nutrient to foundation food IDs and target nutrients
foundation_ids = set(foundation['fdc_id'])
relevant_nutrients = food_nutrient[
    (food_nutrient['fdc_id'].isin(foundation_ids)) &
    (food_nutrient['nutrient_id'].isin(NUTRIENT_IDS.values()))
].copy()

print(f"Relevant nutrient rows: {len(relevant_nutrients)}")

# Pivot to wide format: one row per fdc_id, one column per nutrient
pivot = relevant_nutrients.pivot_table(
    index='fdc_id',
    columns='nutrient_id',
    values='amount',
    aggfunc='first'
).reset_index()

# Rename columns from nutrient IDs to names
id_to_name = {v: k for k, v in NUTRIENT_IDS.items()}
pivot = pivot.rename(columns=id_to_name)

print(f"\nFoods with nutrient data: {len(pivot)}")
print(pivot.head())

Relevant nutrient rows: 1792

Foods with nutrient data: 428
nutrient_id  fdc_id  protein_g  energy_kcal  phosphorus_mg  potassium_mg  \
0            321358       7.35        229.0          166.0         289.0   
1            321359       3.35         50.0          103.0         159.0   
2            321360       0.83         27.0           28.0         260.0   
3            321505        NaN          NaN            0.0           2.0   
4            321611       1.04         21.0           23.0          97.0   

nutrient_id  sodium_mg  
0                438.0  
1                 39.0  
2                  6.0  
3              38700.0  
4                282.0  


In [3]:
# Merge nutrient data with food descriptions
merged = foundation[['fdc_id', 'description']].merge(pivot, on='fdc_id', how='inner')

# Drop foods missing any of the 4 key nutrients (K, P, protein, Na)
required = ['potassium_mg', 'phosphorus_mg', 'protein_g', 'sodium_mg']
merged = merged.dropna(subset=required)

# Fill missing energy with 0 if absent
if 'energy_kcal' not in merged.columns:
    merged['energy_kcal'] = 0
merged['energy_kcal'] = merged['energy_kcal'].fillna(0)

print(f"Foods with complete K/P/protein/Na data: {len(merged)}")

# Remove duplicates by description (some foods have multiple entries)
merged = merged.drop_duplicates(subset='description', keep='first')
print(f"After deduplication: {len(merged)}")

# Sort by description for consistency
merged = merged.sort_values('description').reset_index(drop=True)

# Take up to 400 foods
TARGET_COUNT = 400
if len(merged) > TARGET_COUNT:
    merged = merged.head(TARGET_COUNT)

print(f"\nFinal USDA foods to add: {len(merged)}")
print(merged[['description', 'potassium_mg', 'phosphorus_mg', 'protein_g', 'sodium_mg']].head(10))

Foods with complete K/P/protein/Na data: 383
After deduplication: 336

Final USDA foods to add: 336
                                         description  potassium_mg  \
0                              Almond butter, creamy        745.40   
1      Almond milk, unsweetened, plain, refrigerated         49.27   
2      Almond milk, unsweetened, plain, shelf stable         30.79   
3  Anchovies, canned in olive oil, with salt, dra...        297.80   
4  Apple juice, with added vitamin C, from concen...         95.87   
5                       Apples, fuji, with skin, raw        104.00   
6                       Apples, gala, with skin, raw        106.00   
7               Apples, granny smith, with skin, raw        116.00   
8                 Apples, honeycrisp, with skin, raw         98.00   
9              Apples, red delicious, with skin, raw         95.00   

   phosphorus_mg  protein_g  sodium_mg  
0        506.900  20.787340     0.9963  
1         19.080   0.656250    59.2200  
2     

In [4]:
# KDOQI-aligned thresholds for ckd_stage_safe classification
# Based on potassium content per 100g (primary driver) and phosphorus

def classify_stage_safety(potassium_mg, phosphorus_mg):
    """
    Classify a food's CKD stage safety based on potassium and phosphorus 
    content per 100g, following KDOQI 2020 risk tiers.
    """
    if potassium_mg <= 150 and phosphorus_mg <= 100:
        return '1-5'  # Safe across all stages
    elif potassium_mg <= 250 and phosphorus_mg <= 150:
        return '1-4'  # Restrict at stage 5
    elif potassium_mg <= 350 and phosphorus_mg <= 250:
        return '1-3'  # Avoid/limit at stages 4-5
    elif potassium_mg <= 500:
        return '1-2'  # Only suitable for early CKD
    else:
        return '1'    # High risk - stage 1 only

def generate_note(potassium_mg, phosphorus_mg, stage_safe):
    """Generate a short clinical note matching existing database style."""
    if stage_safe == '1-5':
        return 'Safe all stages - low nutrient content'
    elif stage_safe == '1-4':
        return f'Moderate potassium {potassium_mg:.0f}mg - limit at stage 5'
    elif stage_safe == '1-3':
        return f'High potassium {potassium_mg:.0f}mg - restrict stages 4,5'
    elif stage_safe == '1-2':
        return f'High potassium {potassium_mg:.0f}mg - restrict stages 3,4,5'
    else:
        return f'Very high potassium {potassium_mg:.0f}mg - restrict stages 2,3,4,5'

merged['ckd_stage_safe'] = merged.apply(
    lambda r: classify_stage_safety(r['potassium_mg'], r['phosphorus_mg']), axis=1
)
merged['notes'] = merged.apply(
    lambda r: generate_note(r['potassium_mg'], r['phosphorus_mg'], r['ckd_stage_safe']), axis=1
)

# Print distribution
print("ckd_stage_safe distribution:")
print(merged['ckd_stage_safe'].value_counts())

ckd_stage_safe distribution:
ckd_stage_safe
1-4    88
1-3    86
1-2    83
1-5    45
1      34
Name: count, dtype: int64


In [5]:
# Existing schema columns:
# food_id, english, french, kinyarwanda, category, meal_type, protein_g,
# potassium_mg, phosphorus_mg, sodium_mg, energy_kcal, preparation_method,
# source, ckd_stage_safe, notes

# Determine starting food_id (continue numbering after existing 50)
existing_ids = existing['food_id'].astype(str)
numeric_ids = []
for fid in existing_ids:
    try:
        numeric_ids.append(int(''.join(filter(str.isdigit, str(fid)))))
    except:
        pass
start_id = max(numeric_ids) + 1 if numeric_ids else 51

print(f"New food_id will start at: {start_id}")

# Build new rows
new_rows = []
for i, row in merged.iterrows():
    new_rows.append({
        'food_id': start_id + len(new_rows),
        'english': row['description'],
        'french': '',  # Not available for USDA additions
        'kinyarwanda': '',  # Not available for USDA additions
        'category': 'Other',  # Default - can be refined manually later
        'meal_type': 'Any',
        'protein_g': round(row['protein_g'], 2),
        'potassium_mg': round(row['potassium_mg'], 1),
        'phosphorus_mg': round(row['phosphorus_mg'], 1),
        'sodium_mg': round(row['sodium_mg'], 1),
        'energy_kcal': round(row['energy_kcal'], 1),
        'preparation_method': 'Raw/as analyzed (USDA Foundation Foods)',
        'source': 'USDA FoodData Central (Foundation Foods)',
        'ckd_stage_safe': row['ckd_stage_safe'],
        'notes': row['notes'],
    })

new_df = pd.DataFrame(new_rows)

# Ensure column order matches existing database exactly
new_df = new_df[existing.columns.tolist()]

print(f"\nNew foods dataframe shape: {new_df.shape}")
print(new_df.head(3))

New food_id will start at: 51

New foods dataframe shape: (336, 15)
   food_id                                        english french kinyarwanda  \
0       51                          Almond butter, creamy                      
1       52  Almond milk, unsweetened, plain, refrigerated                      
2       53  Almond milk, unsweetened, plain, shelf stable                      

  category meal_type  protein_g  potassium_mg  phosphorus_mg  sodium_mg  \
0    Other       Any      20.79         745.4          506.9        1.0   
1    Other       Any       0.66          49.3           19.1       59.2   
2    Other       Any       0.55          30.8           29.9       59.6   

   energy_kcal                       preparation_method  \
0          0.0  Raw/as analyzed (USDA Foundation Foods)   
1          0.0  Raw/as analyzed (USDA Foundation Foods)   
2          0.0  Raw/as analyzed (USDA Foundation Foods)   

                                     source ckd_stage_safe  \
0  USDA Foo

In [6]:
# Combine existing 50 with new USDA foods
expanded = pd.concat([existing, new_df], ignore_index=True)

print(f"Original database: {len(existing)} foods")
print(f"USDA additions: {len(new_df)} foods")
print(f"Expanded database: {len(expanded)} foods")

# Save to backend/data/food_database.csv (backup original first)
backup_path = ROOT / 'backend' / 'data' / 'food_database_50_original.csv'
existing.to_csv(backup_path, index=False)
print(f"\nBackup of original 50-food database saved to: {backup_path}")

output_path = ROOT / 'backend' / 'data' / 'food_database.csv'
expanded.to_csv(output_path, index=False)
print(f"Expanded database saved to: {output_path}")

# Summary stats
print(f"\n=== SUMMARY ===")
print(f"Total foods: {len(expanded)}")
print(f"Foods with trilingual names (original 50): {len(existing)}")
print(f"Foods English-only (USDA expansion): {len(new_df)}")
print(f"\nckd_stage_safe distribution (full database):")
print(expanded['ckd_stage_safe'].value_counts())

Original database: 50 foods
USDA additions: 336 foods
Expanded database: 386 foods

Backup of original 50-food database saved to: /Users/jade/GUIDAPLATE/backend/data/food_database_50_original.csv
Expanded database saved to: /Users/jade/GUIDAPLATE/backend/data/food_database.csv

=== SUMMARY ===
Total foods: 386
Foods with trilingual names (original 50): 50
Foods English-only (USDA expansion): 336

ckd_stage_safe distribution (full database):
ckd_stage_safe
1-4    101
1-3     97
1-2     85
1-5     68
1       35
Name: count, dtype: int64


## Summary

The food database has been expanded from 50 to approximately 450 foods.

**Original 50 Rwanda foods** (backed up to `food_database_50_original.csv`):
- Trilingual (English, French, Kinyarwanda)
- Manually validated ckd_stage_safe ratings
- Primary source for food substitution recommendations

**~400 USDA Foundation Foods additions**:
- English names only
- ckd_stage_safe computed automatically from potassium/phosphorus thresholds
  following KDOQI 2020 risk tiers
- category defaulted to 'Other' - can be manually refined for specific 
  categories if needed
- Source: USDA FoodData Central Foundation Foods (whole/raw, unprocessed)

### Next steps
1. Update frontend/src/data/foodDatabase.ts to match new food_database.csv
2. Update Dashboard stat from "50 Rwandan Foods" to new total count
3. Update README food count references
4. Consider manually categorizing the top ~30 most relevant USDA additions
   for better category filtering in Food Explorer